# dataclasses-replace-args — ex1: make a sweep of args variants via dataclasses.replace

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dataclasses-replace-args`. Running the final beacon cell reports progress against the `Config: dataclasses.replace args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: dataclasses.replace args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataclasses-replace-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataclasses-replace-args"
DD_SUBTOPIC = "Config: dataclasses.replace args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Config: `dataclasses.replace(args, ...)` — quick refresher

When you want a *modified copy* of an args object without mutating the original, the idiom is:

```python
from dataclasses import dataclass, replace

@dataclass
class Args:
    lr: float = 1e-3
    bs: int = 32
    epochs: int = 10

args = Args()
args_v2 = replace(args, lr=1e-5)   # only lr changes; bs/epochs copied
```

**`replace` returns a NEW instance.** `args` is untouched. This is the dataclass equivalent of `dict | {'lr': 1e-5}` — same later-wins semantics, but preserves the type and re-runs `__post_init__` validation.

**Why not `args.lr = 1e-5`.** Mutation is fine for one-shot tweaks but breaks any code that holds a reference to the original. The sweep harness, the wandb logger, and the checkpoint writer all expect a stable args object. `replace` gives every consumer its own frozen view.

**Validation re-runs.** Because `replace` calls `Args(**new_fields)`, your `__post_init__` validators fire again — bad overrides raise at replace time, not later when the bad value is used.

### Exercise 1 — make a sweep of args variants via dataclasses.replace

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `dataclasses.replace(base, **overrides)` to produce a sweep of args variants from one base config without mutating the base.
> Keywords: dataclass, replace, immutable-update, sweep
> ```

**KCs targeted:** `dataclasses-replace-keyword-overrides`, `input-isolation-no-mutation`

Implement `ex1_make_lr_sweep(base, lrs)`. The standard sweep-over-LR pattern.

1. `base` is a `TrainingArgs` dataclass instance (already defined in the stub).
2. `lrs` is a list of floats (e.g. `[1e-5, 3e-5, 1e-4]`).
3. For each `lr`, build a *new* `TrainingArgs` with that `lr` and ALL OTHER fields copied from `base` — using `dataclasses.replace(base, lr=lr)`.
4. Return the list of variants in input order.

Constraints:
- Must NOT mutate `base`.
- Each returned variant must be a DIFFERENT object from `base` (not just `base` itself).
- `__post_init__` validation must still run — pass a bad lr and expect `ValueError`.

Output: `list[TrainingArgs]`.

In [ ]:
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')


def ex1_make_lr_sweep(base, lrs):
    return [replace(base, lr=lr) for lr in lrs]


<details><summary>Solution</summary>

```python
from dataclasses import dataclass, replace

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')


def ex1_make_lr_sweep(base, lrs):
    return [replace(base, lr=lr) for lr in lrs]
```

**One-liner is the right scale.** `replace` already does the heavy lifting (copy fields, run `__post_init__`, return a new instance). A list comp wraps it.

**`replace` re-runs `__post_init__`.** That's why the bad `lr=-1.0` test raises — `replace` builds the new instance by calling `TrainingArgs(...)`, which triggers your validator. Free guard against typos in sweep configs.

**Generalizes beyond LR.** Want to sweep `batch_size` instead? `[replace(base, batch_size=bs) for bs in bss]`. Want to sweep two axes? Nested loop, two kwargs to `replace`. The pattern is the same.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()